# Lab 57: The Anchor

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-57.ipynb)

**What you will do:** test the anchoring effect on a friend, compare your result with the classic study, pool the class's data, and check whether a small language model shows the same pull towards an arbitrary number.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 57 you wrote two numbers on slips of paper — **10** and **65** — and showed one slip to
each of two friends before asking: "What percentage of United Nations member states are African
countries?" In the original study (Tversky & Kahneman, 1974), the high-anchor group guessed
around 45%, the low-anchor group guessed around 25%, and the actual answer (about 28%) made no
difference to either group. The anchor pulled the guess towards it even though everyone knew it
was an arbitrary number.

## Record your results

Enter each participant's anchor and estimate below. Add more rows if you tested more than two
people.

In [ ]:
results = pd.DataFrame({
    "participant": ["P1 (anchor 65)", "P2 (anchor 10)"],
    "anchor": [65, 10],
    "estimate_pct": [42, 21],   # example data: replace with your own participants' guesses
})
display(results)
results.plot.bar(x="participant", y="estimate_pct", legend=False, color=["firebrick", "steelblue"], figsize=(5, 3))
plt.ylabel("estimated % African UN member states"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 57 reports that the classic high-anchor group guessed around **45%** and the low-anchor
group guessed around **25%** — a gap of about 20 percentage points, even though the true
answer (about 28%) is the same for everyone.

In [ ]:
high = results.loc[results.anchor == 65, "estimate_pct"].mean()
low = results.loc[results.anchor == 10, "estimate_pct"].mean()
print(f"Your high-anchor estimate: {high:.0f}%   |   published high-anchor value: ~45%")
print(f"Your low-anchor estimate:  {low:.0f}%   |   published low-anchor value:  ~25%")
print(f"Your gap: {high - low:+.0f} points   |   published gap: ~20 points")
if high > low:
    print("Your two participants moved in the direction anchoring predicts.")
else:
    print("Your two participants did not show the predicted direction -- with n=2 this is not"
          " surprising; the effect is reliable on average across many people, not guaranteed"
          " for any one pair.")

## The AI side

The book notes that few-shot examples act as anchors for a language model: earlier
question-answer pairs pull its next answer towards their numbers. Below, a small open model
(DistilGPT-2) is shown two "high" example answers or two "low" example answers before being
asked the African-countries question, and we look at whether its completions drift towards
the priming numbers.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline, set_seed
import re

set_seed(0)
generator = pipeline("text-generation", model="distilgpt2")
question = "What percentage of United Nations member states are located in Africa?"
high_examples = [("What percentage of the Earth's surface is covered by water?", "65%"),
                  ("What percentage of the human body is water?", "70%")]
low_examples = [("What percentage of a lemon's weight is juice?", "10%"),
                 ("What percentage of a small company is a typical minority stake?", "12%")]

def build_prompt(examples):
    text = "".join(f"Q: {q}\nA: {a}\n" for q, a in examples)
    return text + f"Q: {question}\nA:"

def sample_numbers(examples, n=10):
    prompt = build_prompt(examples)
    outs = generator(prompt, max_new_tokens=6, num_return_sequences=n, do_sample=True,
                      temperature=1.0, pad_token_id=generator.tokenizer.eos_token_id)
    nums = []
    for o in outs:
        m = re.search(r"\d{1,3}", o["generated_text"][len(prompt):])
        if m and 0 <= int(m.group()) <= 100:
            nums.append(int(m.group()))
    return nums

high_nums, low_nums = sample_numbers(high_examples), sample_numbers(low_examples)
print("High-anchor completions:", high_nums, "| mean:", f"{pd.Series(high_nums).mean():.0f}" if high_nums else "n/a")
print("Low-anchor completions: ", low_nums, "| mean:", f"{pd.Series(low_nums).mean():.0f}" if low_nums else "n/a")

DistilGPT-2 has 82 million parameters and no real knowledge of UN membership; it is
pattern-matching the digit that tends to follow "A:" given the recent numbers it has seen, not
reasoning about Africa at all. Treat any shift between the two conditions as a loose
illustration of how priming numbers bias a model's output, not as a genuine replication of
human anchoring — small models like this one are unreliable, noisy reasoners, and the model
may sometimes produce no valid number at all.

## Pool the class data

Pool the class's anchor and estimate pairs below (anonymous IDs, not names). The example rows
are placeholders so the cell runs on its own.

In [ ]:
import io
csv_text = """id,anchor,estimate_pct
P01,65,44
P02,10,22
P03,65,50
P04,10,18
P05,65,38
P06,10,27
"""  # example data: replace with your class CSV, e.g. pd.read_csv("anchoring_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
high_boot = [df.loc[df.anchor == 65, "estimate_pct"].sample(frac=1, replace=True).mean() for _ in range(5000)]
low_boot = [df.loc[df.anchor == 10, "estimate_pct"].sample(frac=1, replace=True).mean() for _ in range(5000)]
gap_boot = np.array(high_boot) - np.array(low_boot)
lo, hi = np.percentile(gap_boot, [2.5, 97.5])
print(f"Class mean gap (high anchor - low anchor): {gap_boot.mean():.0f} points "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f}, n = {len(df)})")
print("Published gap for comparison: ~20 points (45% vs 25%)")
df.boxplot(column="estimate_pct", by="anchor", figsize=(5, 3.5))
plt.suptitle(""); plt.ylabel("estimated % African UN member states"); plt.show()

## Questions to think about

1. Did your two participants' estimates move in the direction the anchor predicts? With only n=2, how much should you trust that result either way?
2. The published gap between anchors is about 20 percentage points. How does your pair's gap, and the class's pooled gap, compare?
3. The book notes that the effect persists "even when participants are warned about it." Try warning a new participant that the number is random before asking the question. Does the warning visibly shrink the effect?
4. In the AI demo, did the high-anchor and low-anchor few-shot examples pull DistilGPT-2's completions apart in the same direction as your human data? What would make you trust — or distrust — that result?

## Challenge

Run the "wheel of fortune" version the book suggests: spin a wheel of fortune or generate a
random number on your phone in front of a new participant before asking the African-countries
question, and record their guess. Try it with several people using different random numbers.

In [ ]:
# Example data: replace with your own wheel-of-fortune anchors and estimates.
spins = [12, 88, 45, 30, 70]        # example: the random number the wheel landed on
estimates = [20, 55, 33, 27, 48]    # example: that participant's estimate

spin_df = pd.DataFrame({"spin": spins, "estimate_pct": estimates})
display(spin_df)
corr = spin_df["spin"].corr(spin_df["estimate_pct"])
print(f"Correlation between the random spin and the estimate: r = {corr:.2f}")
print("A positive correlation here would replicate the book's claim that even a transparently")
print("random anchor pulls estimates towards it. The numbers above are example data -- use your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-57.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")